In [ ]:
import matplotlib.pyplot as plt
import dask.array as da
import dask_image.ndfilters
from skimage.measure import label
import tifffile
import skimage as ski
import pandas as pd
import os
import numpy as np
from scipy import ndimage as ndi
import segment_functions as fun
import extract_phenotype

import importlib
importlib.reload(fun)
importlib.reload(extract_phenotype)

In [ ]:
# read in image
root = "/Users/hannahbolen/Desktop/image_analysis/"
img_name = "o8p_day7_s12.ome.tif"
img_path = os.path.join(root, img_name)
img = da.from_array(tifffile.imread(img_path))
# assign channels
FITC = 0
cy5 = 1
# constants -- display
ds = 10
H, W = img.shape
y0 = H//2 - 5120 # underlying "tile" size of 512, make tile excerpt multiple of 512
x0 = W//2 - 5120
# make tile from center of image
nucleiTile = img[FITC][y0:y0+5120, x0:x0+5120]
cy5Tile = img[cy5][y0:y0+5120, x0:x0+5120]
imgTile = img[y0:y0+5120, x0:x0+5120]
# make even smaller tile
nucleiZoom = nucleiTile[1000:3000, 2000:4000]
cy5Zoom = cy5Tile[1000:3000, 2000:4000]
imgZoom = imgTile[1000:3000, 2000:4000]

In [ ]:
# constants -- processing
umPerPx = 0.325
nucleiThreshold = 50
nucleiMin = 150/umPerPx**2
nucleiMax = 12000
nucleiMean = 300/umPerPx**2
smooth=3
cy5_range = (256, 6000)
foci_radius = 5
foci_threshold = 50
# function to visualize
def visualize(base,base_title = "nuclei", clrs=[],**images):
    plt.figure(figsize = (18,18))
    plt.subplot(2, 3, 1)
    plt.axis('off')
    plt.imshow(base, cmap = "gray")
    plt.title(base_title)
    if len(clrs) == len(images):
        for i, img in enumerate(images.keys(), start=2):
            plt.subplot(2,3,i)
            plt.title(img)
            plt.imshow(images[img], cmap = clrs[i])
            plt.axis('off')
            plt.tight_layout()
    elif len(clrs) == 1:
        for i, img in enumerate(images.keys(), start=2):
            plt.subplot(2,3,i)
            plt.title(img)
            plt.imshow(images[img], cmap = clrs[0])
            plt.axis('off')
            plt.tight_layout()
    elif len(clrs) == 0:
        for i, img in enumerate(images.keys(), start=2):
            plt.subplot(2,3,i)
            plt.title(img)
            plt.imshow(images[img])
            plt.axis('off')
            plt.tight_layout()
    else:
        return("Error - incompatible cmap provided")
    plt.show()

In [ ]:
# timer
importlib.reload(fun)
fun.OPS_PROFILE = True
fun.OPS_PROFILE_VERBOSE = True
fun.ops_timing_reset()
# nuclei segmentation arguments
nucleiArgs = dict(threshold=lambda x: nucleiThreshold, 
            area_min=nucleiMin, area_max=nucleiMax,
            smooth=smooth)
# segment nuclei
nuclei = fun.find_nuclei(imgZoom[FITC], **nucleiArgs)
fun.ops_timing_summary()

In [ ]:
print("Extracting phenotype features:")

# Extract features using CellProfiler emulator
phenotype_cp = extract_phenotype.extract_phenotype_cp_multichannel(
    imgZoom,
    nuclei=nuclei,
    cells=None,
    wildcards=None,
    cytoplasms=None,
    foci_channel=cy5,
    channel_names=["FITC", "cy5"],
)

phenotype_cp.to_csv("/Users/hannahbolen/Desktop/image_analysis/imgZoom_featureTable")
phenotype_cp